## Visualization for paper

In [ ]:
import os

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy
import pandas
from xgboost import XGBRegressor

### Data loading

In [ ]:
# Load data and model
total_dataset = pandas.read_parquet(
    "./data/total_dataset.parquet", engine="pyarrow"
)

synthetic_dataset = pandas.read_parquet(
    "./data/synthetic_dataset.parquet", engine="pyarrow"
)

trained_xgb_model = XGBRegressor()
trained_xgb_model.load_model("./data/xgboost_model.bin")

In [ ]:
def plot_cv_timeseries(df_synthetic):
    """Plot the predicted compared vs actual values for a country."""
    country_ground_truth = df_synthetic["load_mw_percentage"].to_numpy()
    country_predictions = df_synthetic["predictions"].to_numpy()
    time_index = pandas.to_datetime(df_synthetic["time_utc"])

    # Create a new figure
    plt.figure(figsize=(15, 8))

    # Plot the ground truth
    plt.plot(
        time_index,
        country_ground_truth,
        label="Ground Truth",
        color="lightgreen",
        alpha=0.7,
    )

    # Plot the predictions
    plt.plot(
        time_index,
        country_predictions,
        label="Predictions",
        color="skyblue",
        alpha=0.7,
    )

    # Plot the absolute difference between predictions and ground truth
    plt.bar(
        time_index,
        numpy.arange(0, len(df_synthetic)),
        abs(country_predictions - country_ground_truth),
        label="Difference",
        color="lightcoral",
        alpha=0.7,
    )

    country_code = df_synthetic["region_code"].iloc[0]
    plt.title(f"{country_code}: Actual vs Predicted Hourly Demand")

    plt.ylabel("Normalized Hourly Demand")
    plt.ylim(
        0,
        round(
            max(
                df_synthetic["load_mw_percentage"].max(),
                df_synthetic["predictions"].max(),
            )
            * 1.5,
            4,
        ),
    )
    plt.legend(loc="upper right")

    # Format x-axis with datetime ticks
    ax = plt.gca()

    # Set major ticks to show years
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # Set minor ticks to show months
    # ax.xaxis.set_minor_locator(mdates.MonthLocator())

    # Rotate labels for better readability
    plt.xticks(rotation=45)

    plt.xlabel("Time")

    # Add grid for better readability
    plt.grid(True, linestyle="--", alpha=0.2)

    # Improve layout
    plt.tight_layout()

    # Save the figure
    plt.savefig(
        os.path.join("./data/" + country_code + "_comparison.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
# Plot the worst performing region CA_AB
plot_cv_timeseries(
    synthetic_dataset[synthetic_dataset["region_code"] == "CA_AB"]
)

In [ ]:
# Plot the best performing region ES
plot_cv_timeseries(synthetic_dataset[synthetic_dataset["region_code"] == "ES"])

### Plot a comparison of two countries for their predictions vs actual values

In [ ]:
syn_country_ES = synthetic_dataset[synthetic_dataset["region_code"] == "ES"]
syn_country_CA_AB = synthetic_dataset[
    synthetic_dataset["region_code"] == "CA_AB"
]

syn_country_ES_2024 = syn_country_ES[
    pandas.to_datetime(syn_country_ES["time_utc"]).dt.year == 2024
]
syn_country_CA_AB_2024 = syn_country_CA_AB[
    pandas.to_datetime(syn_country_CA_AB["time_utc"]).dt.year == 2024
]

country_ES_ground_truth = syn_country_ES_2024["load_mw_percentage"].to_numpy()
country_ES_predictions = syn_country_ES_2024["predictions"].to_numpy()
ES_time_index = pandas.to_datetime(syn_country_ES_2024["time_utc"])

country_CA_AB_ground_truth = syn_country_CA_AB_2024[
    "load_mw_percentage"
].to_numpy()
country_CA_AB_predictions = syn_country_CA_AB_2024["predictions"].to_numpy()
CA_AB_time_index = pandas.to_datetime(syn_country_CA_AB_2024["time_utc"])

In [ ]:
# Create a figure with two subplots side by side
start_time = pandas.to_datetime("2024")
end_time = pandas.to_datetime("2025")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Plot CA_AB data on the left subplot
ax1.plot(
    CA_AB_time_index,
    country_CA_AB_ground_truth,
    label="Ground Truth",
    color="lightgreen",
    alpha=0.7,
)
ax1.plot(
    CA_AB_time_index,
    country_CA_AB_predictions,
    label="Predictions",
    color="skyblue",
    alpha=0.7,
)

# Plot ES data on the right subplot
ax2.plot(
    ES_time_index,
    country_ES_ground_truth,
    label="Ground Truth",
    color="lightgreen",
    alpha=0.7,
)
ax2.plot(
    ES_time_index,
    country_ES_predictions,
    label="Predictions",
    color="skyblue",
    alpha=0.7,
)

ax1.set_title("Alberta, Canada", fontsize=14, weight="bold")
ax1.set_ylabel("Normalized Hourly Demand", fontsize=14)
ax1.set_ylim(0, 0.00025)
ax1.set_xlim(start_time, end_time)

# Format x-axis for ES
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.tick_params(axis="x", rotation=45)
ax1.xaxis.set_minor_locator(mdates.MonthLocator())
ax1.xaxis.set_minor_formatter(
    mdates.DateFormatter("%b")
)  # Show month abbreviations
ax1.grid(True, which="major", linestyle="-", alpha=0.3)  # Year lines (major)
ax1.grid(True, which="minor", linestyle="--", alpha=0.2)  # Month lines (minor)

ax2.set_title("Spain", fontsize=14, weight="bold")
# ax2.set_ylabel("Normalized Hourly Demand")
ax2.set_ylim(0, 0.00025)
ax2.set_xlim(start_time, end_time)

# Format x-axis for CA_AB
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.tick_params(axis="x", rotation=45)
ax2.xaxis.set_minor_locator(mdates.MonthLocator())
ax2.xaxis.set_minor_formatter(
    mdates.DateFormatter("%b")
)  # Show month abbreviations
ax2.grid(True, which="major", linestyle="-", alpha=0.3)  # Year lines (major)
ax2.grid(True, which="minor", linestyle="--", alpha=0.2)  # Month lines (minor)

# Add common x-label
# fig.text(0.515, 0.01, 'Time', ha='center', fontsize=14)

# Add a main title above both plots
fig.suptitle(
    "Worst vs Best Performing Regions on Test Set",
    fontsize=18,
    weight="bold",
    y=0.95,
)

# Add shared legend with bigger markers
handles, labels = ax1.get_legend_handles_labels()
# Modify the handles to have thicker lines for legend
for handle in handles:
    handle.set_linewidth(6)  # Make legend lines thicker
fig.legend(
    handles,
    labels,
    loc="upper right",
    bbox_to_anchor=(0.989, 0.96),  # Top right position
    fontsize=14,
)
for handle in handles:
    handle.set_linewidth(1.5)  # Readjust the width of the lines

# Adjust layout to make room for the legend
plt.tight_layout()
plt.subplots_adjust(top=0.85, bottom=0)

# Save the figure
plt.savefig(
    os.path.join("./data/CA_AB_vs_ES_comparison.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()